# GPT Action 库：Outlook

## 简介

本页面为开发者针对特定应用构建 GPT Action 提供说明与指南。在继续之前，请先熟悉以下内容： 
- [GPT Actions 简介](https://platform.openai.com/docs/actions)
- [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
- [从零构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

这个 GPT Action 概述了如何连接 Outlook——Microsoft 提供的电子邮件与日历事件 Web 服务。该 Action 基于用户的上下文，允许用户在 Outlook 中发送和获取电子邮件及日历事件。

### 价值与业务用例示例

**价值**：用户现在可以利用 ChatGPT 的自然语言能力直接连接 Outlook

**用例示例**： 
- 用户可以查询当天的所有会议，并让 ChatGPT 总结这一天的安排
- 用户可以将 ChatGPT 的输出直接通过电子邮件发送给他人

## 应用信息

### 应用关键链接

开始之前，请先查看该应用的以下链接：
- 应用网站：https://portal.azure.com/
- 应用 API 文档：https://learn.microsoft.com/en-us/graph/api/overview?view=graph-rest-1.0

### 应用前提条件

开始之前，请确保在你的应用环境中完成以下步骤：
- 确保你拥有 [在 Azure 中设置应用注册](https://portal.azure.com/?feature.tokencaching=true&feature.internalgraphapiversion=true#view/Microsoft_AAD_RegisteredApps/ApplicationsListBlade) 的访问权限

## ChatGPT 步骤

### 自定义 GPT 指令 

创建自定义 GPT 后，将下面的文本复制到 Instructions 面板中。有疑问？请查看 [入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

In [ ]:
**Context**: you are specialized GPT designed to manage emails and calendar events through API connections to Microsoft Outlook. This GPT can create, read, send, and alter emails and calendar events based on user instructions. It ensures efficient handling of communication and scheduling needs by leveraging Microsoft Graph API for seamless integration with Outlook services.

**Instructions**:
- When asked to perform a task, use the available actions via the microsoft.graph.com API.
- You should behave professionally and provide clear, concise responses.
- Offer assistance with tasks such as drafting emails, scheduling meetings, organising calendar events, and retrieving email or event details.
- Ask for clarification when needed to ensure accuracy and completeness in fulfilling user requests.
- Always conclude an email by signing off with logged in user's name which can be retrieved via the User.Read endpoint


### OpenAPI Schema 

创建自定义 GPT 后，将下面的文本复制到 Actions 面板中。有疑问？请查看 [入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

In [ ]:
openapi: 3.1.0
info:
  title: Microsoft Graph API Integration
  version: 1.0.0
servers:
  - url: https://graph.microsoft.com/v1.0
components:
  securitySchemes:
    OAuth2:
      type: oauth2
      flows:
        clientCredentials:
          tokenUrl: https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token
          scopes:
            https://graph.microsoft.com/User.Read: Access current user profile
            https://graph.microsoft.com/Mail.Read: Read user mail
            https://graph.microsoft.com/Mail.Send: Send mail
            https://graph.microsoft.com/Calendars.ReadWrite: Read and write user calendars
  schemas:
    UserProfile:
      type: object
      properties:
        id:
          type: string
        displayName:
          type: string
        mail:
          type: string
    UserMessage:
      type: object
      properties:
        id:
          type: string
        subject:
          type: string
        bodyPreview:
          type: string
    CalendarEvent:
      type: object
      properties:
        id:
          type: string
        subject:
          type: string
        start:
          type: object
          properties:
            dateTime:
              type: string
            timeZone:
              type: string
        end:
          type: object
          properties:
            dateTime:
              type: string
            timeZone:
              type: string
    NewEvent:
      type: object
      properties:
        subject:
          type: string
        start:
          type: object
          properties:
            dateTime:
              type: string
            timeZone:
              type: string
        end:
          type: object
          properties:
            dateTime:
              type: string
            timeZone:
              type: string
        attendees:
          type: array
          items:
            type: object
            properties:
              emailAddress:
                type: object
                properties:
                  address:
                    type: string
                  name:
                    type: string
    SendMailRequest:
      type: object
      properties:
        message:
          type: object
          properties:
            subject:
              type: string
            body:
              type: object
              properties:
                contentType:
                  type: string
                content:
                  type: string
            toRecipients:
              type: array
              items:
                type: object
                properties:
                  emailAddress:
                    type: object
                    properties:
                      address:
                        type: string
security:
  - OAuth2: []
paths:
  /me:
    get:
      operationId: getUserProfile
      summary: Get the authenticated user's profile
      security:
        - OAuth2: []
      responses:
        '200':
          description: A user profile
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/UserProfile'
  /me/messages:
    get:
      operationId: getUserMessages
      summary: Get the authenticated user's messages
      security:
        - OAuth2: []
      parameters:
        - name: $top
          in: query
          required: false
          schema:
            type: integer
            default: 10
            description: Number of messages to return
        - name: $filter
          in: query
          required: false
          schema:
            type: string
            description: OData filter query to narrow results
        - name: $orderby
          in: query
          required: false
          schema:
            type: string
            description: OData order by query to sort results
      responses:
        '200':
          description: A list of user messages
          content:
            application/json:
              schema:
                type: array
                items:
                  $ref: '#/components/schemas/UserMessage'
  /me/sendMail:
    post:
      operationId: sendUserMail
      summary: Send an email as the authenticated user
      security:
        - OAuth2: []
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/SendMailRequest'
      responses:
        '202':
          description: Accepted
  /me/events:
    get:
      operationId: getUserCalendarEvents
      summary: Get the authenticated user's calendar events
      security:
        - OAuth2: []
      responses:
        '200':
          description: A list of calendar events
          content:
            application/json:
              schema:
                type: array
                items:
                  $ref: '#/components/schemas/CalendarEvent'
    post:
      operationId: createUserCalendarEvent
      summary: Create a new calendar event for the authenticated user
      security:
        - OAuth2: []
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/NewEvent'
      responses:
        '201':
          description: Created
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/CalendarEvent'



## 身份验证说明

以下是在 Outlook 中设置身份验证的说明。有疑问？请查看 [入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。


### Azure 步骤

1. <b>应用注册</b>：第一步是在 [Azure Portal](https://portal.azure.com) 中注册一个新的应用，用于在我们的应用与 Azure Active Directory/Entra ID 之间集成 OAuth。只需为应用起一个合适的名称，Redirect URI 暂时留空（稍后会回来设置），然后保存。

![gptactions_outlook_registerapplication.png](../../../images/gptactions_outlook_registerapplication.png)

2. <b>证书和密码</b>：接下来需要生成一个客户端密码（client secret），用于 GPT 与 Azure 之间的安全通信。在应用注册中，从侧边栏导航到 <b>Certificate & secrets</b>

![gptactions_outlook_secrets.png](../../../images/gptactions_outlook_secrets.png)

点击 New client secret，使用所需的名称和过期日期创建一个新的客户端密码。点击保存后会得到一个 Secret，供创建 GPT 时使用。请务必保存 **Value** 字段，因为它只在创建时可见，而我们稍后会用到它！

![gptactions_outlook_secretvalue.png](../../../images/gptactions_outlook_secretvalue.png)

3. <b>API 权限</b>：下一步是为集成授予执行所需操作的权限范围（scope）。 
在应用注册中，从侧边栏导航到 <b>Manage > API permissions</b>。 

![gptactions_outlook_permissions.png](../../../images/gptactions_outlook_permissions.png)

点击 <b>Add a permission</b>，并在打开的侧边菜单中选择 <b>Microsoft graph > Delegated Permissions</b>。使用搜索栏添加以下权限：
- Calendars.ReadWrite
- Mail.Read
- Mail.Send
- User.Read


![gptactions_outlook_permissionadd.png](../../../images/gptactions_outlook_permissionadd.png)

### 在 ChatGPT 中

在 ChatGPT 中，点击“Authentication”并选择 **“OAuth”**。填写以下信息。 

- **Client ID**：Azure 已注册应用 Overview 页面上 **Application (client) ID** 下列出的值
- **Client Secret**：在 **Azure 步骤**第 2 步中保存的密码 **Value**

对于下面两项输入，请将 <Tenant_ID> 替换为已注册应用 Overview 页面上 **Directory (tenant) ID** 下列出的值

- **Authorization URL**：https://login.microsoftonline.com/<Tenant_ID>/oauth2/v2.0/authorize
- **Token URL**：https://login.microsoftonline.com/<Tenant_ID>/oauth2/v2.0/token
- **Scope**：https://graph.microsoft.com/User.Read https://graph.microsoft.com/Mail.Send https://graph.microsoft.com/Mail.Read https://graph.microsoft.com/Calendars.ReadWrite 
- **Token Exchange Method**：Default (POST Request)

### Action 后续步骤

在 ChatGPT 中设置好身份验证后，请在应用中按以下步骤完成 Action 的配置。 

- 从 GPT Action 中复制回调 URL

![gptactions_outlook_callback.png](../../../images/gptactions_outlook_callback.png)

- 在 Azure 应用中，导航到 **Manage > Authentication** 选项卡，点击 **Add a platform**，选择 **Web**，并在 **Redirect URI** 下添加你的回调 URL


![gptactions_outlook_redirectconfig.png](../../../images/gptactions_outlook_redirectconfig.png)
![gptactions_outlook_redirectinput.png](../../../images/gptactions_outlook_redirectinput.png)

### 常见问题与故障排查

- **回调 URL 错误**：如果在 ChatGPT 中遇到回调 URL 错误，请再次检查 Callback URL 的值，因为它偶尔会随身份验证配置的更改而变化


*有希望我们优先支持的集成吗？我们的集成中有错误吗？请在我们的 GitHub 中提交 PR 或 issue，我们会及时查看。*
